# Qwen3-ASR Transcription

A lightweight pipeline to transcribe audio files to text using [Qwen3-ASR](https://huggingface.co/Qwen/Qwen3-ASR-0.6B).

**Steps:**
1. Open Google Colab (T4 GPU)
2. Install dependencies
3. Load the ASR model
4. Upload an audio file
5. Transcribe
6. Save the output

## Block 1 — Open Google Colab with T4 GPU

Before running any code, make sure you're using a GPU runtime:
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** under "Hardware accelerator"
3. Click **Save**

Why GPU? The ASR model runs much faster on GPU. CPU works but is significantly slower.

## Block 2 — Install dependencies

Install the ASR package and audio processing libraries. The `-q` flag keeps output minimal.

In [ ]:
# Install the official ASR Python package that supports model loading.
!pip install -q qwen-asr

# Install audio processing and torch
!pip install -q torch torchaudio librosa soundfile

## Block 3 — Import modules and check GPU

Import the required libraries and verify that Colab assigned us a GPU. If `device` prints `cpu`, the model will still work but much slower.

In [ ]:
import torch
import librosa
import soundfile as sf

from qwen_asr import Qwen3ASRModel

# Check if GPU is available, fall back to CPU if not
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## Block 4 — Load the Qwen3-ASR model

Choose `Qwen/Qwen3-ASR-0.6B` (faster, lower VRAM) or `Qwen/Qwen3-ASR-1.7B` (higher accuracy, more VRAM).

- `bfloat16` is used on GPU to save memory with minimal accuracy loss
- `float32` is used on CPU since bfloat16 isn't supported there
- `device_map="auto"` lets the library place the model on the GPU automatically

In [ ]:
# Switch to "Qwen/Qwen3-ASR-1.7B" for higher accuracy (needs more VRAM)
model_name = "Qwen/Qwen3-ASR-0.6B"

asr_model = Qwen3ASRModel.from_pretrained(
    model_name,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

print("ASR model loaded:", model_name)

## Block 5 — Upload and play the audio file

Upload any audio file (.wav, .mp3, .m4a, etc.). The file will play back in Colab so you can verify it loaded correctly.

Tip: Use a short audio clip (under 2 minutes) for faster results.

In [ ]:
from google.colab import files
from IPython.display import Audio, display

# Opens a file picker in Colab — select your audio file
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]

print("Uploaded audio:", audio_file)

# Play the audio so you can verify it loaded correctly
display(Audio(audio_file))

## Block 6 — Convert to mono 16kHz

The Qwen3-ASR model requires audio in 16kHz mono format. This step resamples and converts any input audio to match.

- `librosa.load()` handles format conversion and resampling automatically
- The cleaned audio is saved as `clean_audio.wav` for the next step

In [ ]:
# Resample to 16kHz mono — the model's required input format
audio, sr = librosa.load(audio_file, sr=16000, mono=True)

clean_path = "clean_audio.wav"
sf.write(clean_path, audio, 16000)

print("Normalized audio saved as:", clean_path)

## Block 7 — Transcribe the audio

Run the ASR model on the cleaned audio. Key options:

- `language=None` — automatic language detection (supports 20+ languages)
- `return_time_stamps=False` — set to `True` if you want word-level timing
- Results come back as a list; `results[0].text` gives the transcription string

In [ ]:
# Run transcription — language=None enables automatic language detection
results = asr_model.transcribe(
    audio=clean_path,
    language=None,
    return_time_stamps=False
)

# Extract the text from the first (and only) result
transcript = results[0].text

print("Transcription output:")
print(transcript)

## Block 8 — Save reference text

Save the transcription to a text file. This is useful for:
- Voice cloning (the reference text pairs with the audio)
- Archiving transcriptions for later analysis
- Using as training data for other NLP tasks

In [ ]:
# Save transcription to file
with open("reference_text.txt", "w") as f:
    f.write(transcript)

print("Reference text saved to reference_text.txt")